# 03 · C: 기존 R1 동결
B와 같은 원본 R1을 학습 없이 평가하는 비교군의 설명용 노트북입니다. 사용자 실행은 04에 통합되어 별도 실행은 생략합니다. optimizer나 새 출력층을 만들지 않습니다. 기존 정책을 유지하며 seed 0은 단일 참조 예측을 식별하는 표기입니다.

## 공통 준비
개인 `alexyoo0506@gmail.com` Colab Pro와 해당 계정의 Drive를 사용합니다. 사용자가 Drive 5TB 보유를 확인했으므로 저장용량 확인·입력 절차는 생략합니다. Drive 연결 시 해당 계정을 선택합니다. GPU 작업은 T4 런타임을 선택하고, Drive 연결은 Google 화면에서 직접 승인합니다. GCP·Vertex 제출 또는 GCS 업로드 코드는 없습니다.

현재 파일은 아직 원격 main에 게시되지 않았으므로 main을 clone해서 실행하지 않습니다. 제공한 실행 패키지의 `transfer-experiments` 폴더를 개인 Drive의 `MyDrive/LLM_Router/`에 업로드하세요. 경로·SHA·데이터·source pin이 모두 채워져 있으므로 수정 없이 전체 실행합니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

BASE = Path("/content/drive/MyDrive/LLM_Router/transfer-experiments")
SOURCE_ZIP = BASE / "source" / "transfer-source.zip"
EXPECTED_SOURCE_SHA256 = SOURCE_ZIP.with_suffix(".sha256").read_text().split()[0]
RUN_ID = "r1-to-r0-transfer-v2-run001"  # A/B/C 모두 같은 ID. 새 계획에는 새 ID
RUN = BASE / "runs" / RUN_ID
INPUTS = BASE / "inputs" / "experiment-A"
PINS = INPUTS / "source_pins.json"
DATA = INPUTS / "data"
LOG = INPUTS / "feedback-log.parquet"

assert len(EXPECTED_SOURCE_SHA256) == 64, "소스 ZIP의 검증된 SHA-256을 입력하세요"
assert hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest() == EXPECTED_SOURCE_SHA256
REPO = Path("/content/router-transfer-source")
with zipfile.ZipFile(SOURCE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
for name, expected in json.loads((REPO / "source-bundle-manifest.json").read_text()).items():
    assert hashlib.sha256((REPO / name).read_bytes()).hexdigest() == expected, name
EXPERIMENT = REPO / "ml/notebooks/r1_to_r0_transfer"
HELPER = EXPERIMENT / "experiment.py"
CONFIG = EXPERIMENT / "experiment_config.json"
ENV = dict(os.environ, PYTHONPATH=str(REPO / "ml/src"))
def run(*args):
    subprocess.run([sys.executable, str(HELPER), *map(str, args)], env=ENV, check=True)
print("Shared run directory:", RUN)


## 패키지 설치
Colab 메뉴 **런타임 → 런타임 유형 변경 → 런타임 버전 2026.07**(Python 3.12.13), GPU T4를 선택합니다. 기본 최신 버전은 사용하지 않습니다. Python 3.11/3.12와 CUDA 12.4용 PyTorch 2.5.1을 사용합니다. 설치 후 이미 import한 패키지 버전이 다르면 Colab 세션을 재시작하고 경로 설정 셀부터 다시 실행하세요. 학습 helper가 런타임 계약 불일치를 검사합니다.

In [ ]:
assert sys.version_info[:2] in {(3, 11), (3, 12)}, (f"현재 Python {sys.version.split()[0]}. Colab 런타임 → 런타임 유형 변경 → 런타임 버전을 2026.07(Python 3.12)로 바꾸고 처음부터 모두 실행하세요. T4는 유지합니다.")
# Text-only model: remove preinstalled optional packages incompatible with pinned torch.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "torchaudio"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.5.1", "--index-url", "https://download.pytorch.org/whl/cu124"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(EXPERIMENT / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO / "ml")], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요"
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)


In [ ]:
# C 평가는 04_compare_results가 자동 수행하므로 이 노트북을 별도로 실행할 필요가 없습니다.


C의 최종 보호 데이터 평가는 A/B 모든 seed의 학습 완료 이후 개발검증 선택을 자동 고정한 뒤 04_compare_results에서 수행합니다. RouterArena에는 대응 후보 비용이 없으므로 cost-free argmax를 **난이도 정렬 진단**으로만 보고합니다.